# Projeto 01: Crédito Inteligente

<h3>Integrantes:</h3> Leonardo Nakashima, João Pedro Paulino, Isadora Jardim, Jefferson Lopes e Maria Eduarda

## Importando as bibliotecas 

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, recall_score, f1_score, classification_report
from datetime import datetime
import openpyxl
import os

ModuleNotFoundError: No module named 'sklearn'

# Criando o dataFrame com o credito_banco.csv

In [ ]:
df = pd.read_csv('credito_banco.csv')

## ETAPA 1: Limpeza dos dados com pandas

In [ ]:
print("=== ETAPA 1: Limpeza dos dados com pandas ===\n")

print(f"Dataset original: {df.shape}")
print(f"Valores ausentes:\n{df.isnull().sum()}")

# Removend valores nulos
df_clean = df.dropna()
print(f"\nApós remover valores ausentes: {df_clean.shape}")

# Removendo rendas outliers
Q1 = df_clean['renda_anual'].quantile(0.25)
Q3 = df_clean['renda_anual'].quantile(0.75)
IQR = Q3 - Q1
limite_inf = Q1 - 1.5 * IQR
limite_sup = Q3 + 1.5 * IQR

outliers = df_clean[(df_clean['renda_anual'] < limite_inf) | (df_clean['renda_anual'] > limite_sup)]
print(f"Outliers de renda detectados: {len(outliers)}")

df_clean = df_clean[(df_clean['renda_anual'] >= limite_inf) & (df_clean['renda_anual'] <= limite_sup)]
print(f"Dataset após remover outliers: {df_clean.shape}")

print(f"\nDistribuição de inadimplência:")
print(f"  Adimplentes: {(df_clean['inadimplente'] == 0).sum()}")
print(f"  Inadimplentes: {(df_clean['inadimplente'] == 1).sum()}")
print(f"  Taxa: {df_clean['inadimplente'].mean()*100:.1f}%")


## ETAPA 3: Treinar Robô e Automatizar a exportação.

In [ ]:
print("\n=== ETAPA 3.1: Treinando o Modelo de Machine Learning ===")

X = df_clean[['renda_anual', 'idade', 'divida_atual', 'score_credito']]
y = df_clean['inadimplente']

#definição de Features e Target
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

#treina o modelo
modelo_rf = RandomForestClassifier(n_estimators=100, random_state=42)
modelo_rf.fit(X_train, y_train)

y_pred = modelo_rf.predict(X_test)

#calcula o Recall e o F1-score da classe 1
acuracia = accuracy_score(y_test, y_pred)
recall_inadimplente = recall_score(y_test, y_pred)
f1_inadimplente = f1_score(y_test, y_pred)

print(f"Acurácia Geral: {acuracia:.2%}")
print(f"Recall (Classe 1 - Inadimplentes): {recall_inadimplente:.2%}")
print(f"F1-Score (Classe 1 - Inadimplentes): {f1_inadimplente:.2%}")

if acuracia >= 0.85:
    print("\nObjetivo de acurácia (>= 85%) atingido!")
else:
    print("\nAtenção: A acurácia ficou abaixo de 85%. Considere ajustar os hiperparâmetros.")

print("\nRelatório de Classificação Detalhado:")
print(classification_report(y_test, y_pred))

#-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-#
print("\n=== ETAPA 3.2: Automatização RPA (Exportação) ===")

#aplica o modelo na base de dados limpa
df_clean['previsao_inadimplencia'] = modelo_rf.predict(df_clean[['renda_anual', 'idade', 'divida_atual', 'score_credito']])

lista_alto_risco = df_clean[df_clean['previsao_inadimplencia'] == 1]

data_atual = datetime.now().strftime("%Y-%m-%d")
nome_arquivo = f"relatorio_risco_{data_atual}.xlsx"

try:
    lista_alto_risco.to_excel(nome_arquivo, index=False)
    print(f"Sucesso! Relatório '{nome_arquivo}' gerado com {len(lista_alto_risco)} clientes de alto risco.")
    print(f"Caminho do arquivo: {os.path.abspath(nome_arquivo)}")
except Exception as e:
    print(f"Erro ao exportar:  {e}")